# 第 31 课｜真实小模型低比特加载与回归评测

**FP16 → bitsandbytes INT8 / NF4 → 实际模块核验 → 显存记录 → 同题配对比较**

这一课把第30课的量化数学接到真实加载后端，并原样复用第29课的题集、提示和评分器。
你将得到同一固定版本模型的三路实验，而不是把不同来源的模型分数拼在一起。

**交付范围：** 本 Notebook 自包含；同名 `.py` 也是独立入口，两者任选其一。默认只执行离线实验，不联网、不自动安装库、不加载模型。不生成完整学习包。

**制作时验证边界：** 当前制作环境为 CPU，未安装 Transformers、Accelerate、bitsandbytes，且无法访问依赖/模型下载服务。已保存输出只来自实际执行的离线代码。真实模型、CUDA量化内核、显存和速度数据均须在你的电脑上运行；没有填入虚构成绩。

**先修：** 第28课模板/生成接口，第29课评分，第30课量化误差；第19、22–24课的缓存、显存和dtype知识。

**学习目标：** 解释“存储位宽、计算dtype、缓存dtype”三个层次；确认低比特模块真被加载；不让量化失败混入质量排名；用固定题目定位回归。

## 0. 本课怎么使用

先按顺序运行离线部分，再选择开启真实加载。标为“实现”的长代码单元是完整工具代码，默认折叠；可以先运行，再根据对应章节展开阅读。

离线环境仅用 Python 标准库；安装了 PyTorch 时，额外运行张量/缓存接口检查。真实三路实验**限定单张CUDA卡**，是为了控制本课比较条件，不是在声称 bitsandbytes 只能在CUDA运行。

不要先尝试更大模型。继续使用第28课的 `Qwen/Qwen3-0.6B`，固定仓库 commit，并让CPU卸载保持关闭。这样能先区分环境、数值和评测问题。

**重要：** 各模式会启动新进程，但其他Notebook内核或应用仍可能占GPU。开始真实实验前，关闭旧模型或重启内核，不要同时运行其他GPU任务。

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""第31课：真实小模型低比特加载与配对回归评测。
默认仅运行离线检查与教学演示；不会下载、不加载真实模型。
真实路径需显式 --run，并限定单张 CUDA GPU；FP16/INT8/NF4 各用新进程。
第29课题集、提示与严格评分保持不变。本课不执行模型生成的代码。
"""
from __future__ import annotations

# %% 01. 标准库工具与固定协议
import argparse
import copy
import gc
import hashlib
import importlib.metadata
import importlib.util
import json
import math
import platform
import re
import statistics
import subprocess
import sys
import tempfile
import time
import traceback
from collections import Counter
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any


MODEL_ID = "Qwen/Qwen3-0.6B"

REVISION = "c1899de289a04d12100db370d81485cdf75e47ca"

FORMAT_VERSION = "lesson29-evaluation-v1"

GRADER_VERSION = "strict-ascii-int-label-json-v1"

TASKS = ("arithmetic", "code_trace", "routing", "extraction")

SYSTEM_PROMPT = "你正在完成固定题集。请严格遵守每道题的输出格式，只输出最终答案，不解释、不使用 Markdown 代码围栏。"

CONTRACTS = {
    "arithmetic": "只输出一个十进制整数。使用 ASCII 数字和必要的负号，不加单位、前导零、加号或解释。",
    "code_trace": "判断下面 Python 3 代码打印的整数。只输出该整数，不执行任何外部操作，不加解释或代码围栏。",
    "routing": "使用字面规则分类：文本含“退款”则输出 PAYMENT；否则含“快递”则输出 DELIVERY；否则输出 OTHER。只输出一个大写标签。",
    "extraction": '只输出一个 JSON 对象，恰好包含 "item" 和 "quantity" 两个字段。item 为名称字符串，quantity 为非负整数。不得增加字段、解释或代码围栏。',
}


LESSON_FORMAT = "lesson31-quantized-regression-v1"
MODES = ("fp16", "int8", "nf4")
SOURCES = {
    "bnb": "https://huggingface.co/docs/transformers/v4.57.3/en/quantization/bitsandbytes",
    "install": "https://huggingface.co/docs/bitsandbytes/installation",
    "nf4": "https://arxiv.org/abs/2305.14314",
    "int8": "https://arxiv.org/abs/2208.07339",
    "linear4bit": "https://huggingface.co/docs/bitsandbytes/reference/nn/linear4bit",
    "model": f"https://huggingface.co/{MODEL_ID}/blob/{REVISION}/README.md",
    "memory": "https://docs.pytorch.org/docs/stable/notes/cuda.html#memory-management",
    "generation": "https://huggingface.co/docs/transformers/v4.57.3/en/main_classes/text_generation",
}


def fingerprint(value: Any) -> str:
    text = json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False)
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def json_write(path: Path, value: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")

def positive_int(value: Any, name: str) -> int:
    if type(value) is not int or value <= 0:
        raise ValueError(f"{name} 必须是正整数，不接受 bool")
    return value

def unique_run_dir(root: str | Path, prefix: str) -> Path:
    root = Path(root).expanduser()
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    result = root / f"{prefix}_{stamp}"
    result.mkdir(parents=True, exist_ok=False)
    return result

## 1. 同源三路对照：这些名字不是可互换的 dtype

| 模式 | 本课改变什么 | 本课没有声称什么 |
|---|---|---|
| `fp16` | 普通权重以FP16加载，作为本次基线 | 不是原始训练过程的完整精度复现 |
| `int8` | 用 LLM.int8 的 `Linear8bitLt` 替换投影 | 不是把整网一律 `.to(int8)` |
| `nf4` | 用 `Linear4bit`、NF4码表、双重量化，计算dtype为FP16 | 不是第30课均匀INT4，也不是全程4位算术 |

LLM.int8 会结合量化计算与较高精度的离群特征处理。[2]
NF4 用非均匀码表表示归一化后的权重；双重量化还会量化量化常数，以减少元数据开销。[3]

**这里只做推理，不是QLoRA训练。** QLoRA还涉及冻结量化基座、训练低秩适配器；本课没有创建优化器，没有更新参数。

为什么不用“4位一定比8位差”预先给结果排座次？两者不只是位数不同，算法和处理路径也不同；应先定义比较条件，再检查数据。

In [2]:
# %% 04. 本课设置与量化计划：这些纯函数无需加载第三方库
@dataclass(frozen=True)
class Settings:
    split: str = "dev"
    gpu: int = 0
    max_new: int = 64
    max_input: int = 512
    context_limit: int = 2048
    seed: int = 29
    local_files_only: bool = False
    benchmark: bool = False
    benchmark_steps: int = 16
    benchmark_repeats: int = 5
    output_dir: str = "lesson_31_outputs"


def validate_settings(s: Settings) -> None:
    if s.split not in {"dev", "eval"}:
        raise ValueError("split 必须为 dev/eval")
    if type(s.gpu) is not int or s.gpu < 0:
        raise ValueError("gpu 必须是非负整数")
    for name in ("max_new", "max_input", "context_limit", "benchmark_steps", "benchmark_repeats"):
        positive_int(getattr(s, name), name)
    if s.max_input + s.max_new > s.context_limit:
        raise ValueError("本课要求 max_input + max_new <= context_limit")
    if s.benchmark_steps < 2:
        raise ValueError("benchmark_steps 至少为2：prefill预测一次，其余为decode")
    if type(s.seed) is not int or s.seed < 0:
        raise ValueError("seed 必须是非负整数")


def quantization_plan(mode: str) -> dict | None:
    """字符串 dtype 在真正构造 BitsAndBytesConfig 时再转为 torch.dtype。"""
    if mode not in MODES:
        raise ValueError("未知模式；仅接受 fp16/int8/nf4")
    if mode == "fp16":
        return None
    if mode == "int8":
        return {"load_in_8bit": True, "llm_int8_threshold": 6.0,
                "llm_int8_enable_fp32_cpu_offload": False,
                "llm_int8_has_fp16_weight": False,
                "llm_int8_skip_modules": ["lm_head"]}
    return {"load_in_4bit": True, "bnb_4bit_quant_type": "nf4",
            "bnb_4bit_use_double_quant": True,
            "bnb_4bit_compute_dtype": "float16",
            "bnb_4bit_quant_storage": "uint8",
            "llm_int8_skip_modules": ["lm_head"],
            "llm_int8_enable_fp32_cpu_offload": False}


def make_quant_config(mode, torch_module, config_factory):
    plan = quantization_plan(mode)
    if plan is None:
        return None
    for key in ("bnb_4bit_compute_dtype", "bnb_4bit_quant_storage"):
        if key in plan:
            plan[key] = getattr(torch_module, plan[key])
    return config_factory(**plan)


def generation_options(s: Settings, eos: list[int], pad_id: int) -> dict:
    validate_settings(s)
    if not eos or any(type(i) is not int or i < 0 for i in eos):
        raise ValueError("EOS 列表不合法")
    if type(pad_id) is not int or pad_id < 0:
        raise ValueError("PAD ID 不合法")
    return {"max_new_tokens": s.max_new, "do_sample": False, "num_beams": 1,
            "num_return_sequences": 1, "use_cache": True,
            "cache_implementation": "dynamic", "repetition_penalty": 1.0,
            "eos_token_id": list(dict.fromkeys(eos)), "pad_token_id": pad_id,
            "return_dict_in_generate": True, "output_scores": False}


def required_projections(layer_count: int) -> list[str]:
    positive_int(layer_count, "layer_count")
    return [f"model.layers.{i}.{part}.{name}" for i in range(layer_count)
            for part, names in (("self_attn", ("q_proj", "k_proj", "v_proj", "o_proj")),
                                ("mlp", ("gate_proj", "up_proj", "down_proj")))
            for name in names]


def validate_projection_inventory(rows: list[dict], mode: str, layer_count: int) -> None:
    """验收所需Qwen3投影，不把仅设置标志当作量化成功。"""
    if mode not in MODES:
        raise ValueError("未知模式")
    by_name = {r["name"]: r for r in rows}
    if len(by_name) != len(rows):
        raise ValueError("模块名称重复")
    expected = {"fp16": "Linear", "int8": "Linear8bitLt", "nf4": "Linear4bit"}[mode]
    expected_dtype = {"fp16": "torch.float16", "int8": "torch.int8", "nf4": "torch.uint8"}[mode]
    for name in required_projections(layer_count):
        if name not in by_name:
            raise ValueError(f"缺少预期投影：{name}")
        row = by_name[name]
        if row["kind"] != expected or row["weight_dtype"] != expected_dtype:
            raise ValueError(f"投影没有按计划加载：{name}, {row['kind']}, {row['weight_dtype']}")
        if mode == "nf4":
            if not row.get("has_quant_state") or row.get("quant_type") != "nf4":
                raise ValueError(f"缺少真实 NF4 量化状态：{name}")
            if row.get("compute_dtype") != "torch.float16" or not row.get("nested"):
                raise ValueError(f"NF4计算精度/双重量化设置不符：{name}")
        if mode == "int8" and row.get("has_fp16_weights") is not False:
            raise ValueError(f"INT8仍保留可训练FP16主权重：{name}")
    if "lm_head" in by_name and by_name["lm_head"]["kind"] != "Linear":
        raise ValueError("本课显式保留普通 lm_head，不把输出头改成量化层")


def assert_single_gpu_devices(devices: dict[str, str], gpu: int) -> None:
    if not devices or any(d != f"cuda:{gpu}" for d in devices.values()):
        bad = {n: d for n, d in devices.items() if d != f"cuda:{gpu}"}
        raise ValueError(f"本课禁止CPU/disk/meta/其他GPU回退：{bad}")


def ideal_storage_demo() -> dict:
    """纯假设，不是Qwen显存实测。量化权重占80%，其余20%维持FP16。"""
    total, quantized = 600_000_000, 480_000_000
    rest = total - quantized
    return {"kind": "analytical_example_not_model_measurement", "parameters": total,
            "quantized_linear_parameters": quantized,
            "fp16_weight_bytes": total * 2,
            "int8_codes_plus_fp16_rest_bytes": quantized + rest * 2,
            "four_bit_codes_plus_fp16_rest_bytes": quantized // 2 + rest * 2,
            "excluded": ["scales", "nested metadata", "KV cache", "activations", "allocator", "CUDA context"]}


def memory_numbers(start: int, live: int, peak: int, reserved: int) -> dict:
    if any(type(x) is not int or x < 0 for x in (start, live, peak, reserved)):
        raise ValueError("显存数值必须是非负整数")
    if peak < max(start, live) or reserved < live:
        raise ValueError("显存测量顺序或口径不一致")
    return {"start_allocated_bytes": start, "end_allocated_bytes": live,
            "peak_allocated_bytes": peak, "peak_increment_bytes": peak - start,
            "end_reserved_bytes": reserved,
            "scope": "PyTorch CUDA allocator; not full process / nvidia-smi"}

In [3]:
for mode in MODES:
    print(mode, json.dumps(quantization_plan(mode), ensure_ascii=False, indent=2))

fp16 null
int8 {
  "load_in_8bit": true,
  "llm_int8_threshold": 6.0,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": [
    "lm_head"
  ]
}
nf4 {
  "load_in_4bit": true,
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "bnb_4bit_compute_dtype": "float16",
  "bnb_4bit_quant_storage": "uint8",
  "llm_int8_skip_modules": [
    "lm_head"
  ],
  "llm_int8_enable_fp32_cpu_offload": false
}


### 1.1 读懂这几个参数

```python
BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_storage=torch.uint8,
)
```

`uint8` 是存放打包码的容器，一字节可以包含两个四位码；不能看到 `uint8` 就说“加载成8位了”。`float16` 是计算设置，不是说所有权重已经展开并永久存成FP16。[1][4]

**也不要只检查 `next(model.parameters()).dtype`。** 第一个参数可能来自未量化Embedding，不能代表所有Linear；`.numel()` 还可能数到的是打包后的容器元素，不能直接当原始逻辑参数量。

## 2. 为什么四位模型的显存不保证是FP16的四分之一？

先做一个**纯假设**：6亿个参数，其中80%可量化，20%继续使用FP16。

不计尺度、元数据和运行时开销，FP16权重为12亿字节；只把80%改成4位码后，码与剩余FP16权重合计仍有4.8亿字节，而不是3亿字节。

真实模型还包括量化常数、未量化层、KV cache、激活、临时工作区和分配器保留空间。本课不会把这个假设套成Qwen的实测值。

In [4]:
storage = ideal_storage_demo()
print(json.dumps(storage, ensure_ascii=False, indent=2))
print("假设的有效载荷比：", storage["four_bit_codes_plus_fp16_rest_bytes"] / storage["fp16_weight_bytes"])

{
  "kind": "analytical_example_not_model_measurement",
  "parameters": 600000000,
  "quantized_linear_parameters": 480000000,
  "fp16_weight_bytes": 1200000000,
  "int8_codes_plus_fp16_rest_bytes": 720000000,
  "four_bit_codes_plus_fp16_rest_bytes": 480000000,
  "excluded": [
    "scales",
    "nested metadata",
    "KV cache",
    "activations",
    "allocator",
    "CUDA context"
  ]
}
假设的有效载荷比： 0.4


### 2.1 四种内存口径要分开

| 记录 | 表示什么 | 不等于什么 |
|---|---|---|
| `get_memory_footprint` | 库的参数/buffer核算 | 不保证涵盖全部量化元数据、KV和工作区 |
| `memory_allocated` | 当前PyTorch分配器统计的活跃GPU张量内存 | 不是整个进程的显存 |
| `max_memory_allocated` | 从上次reset起的峰值 | 必须说明测量阶段 |
| `memory_reserved` | 分配器保留的内存 | 不等于活跃张量占用 |

本课分别记录加载阶段和生成阶段。生成峰值减去开始时的allocated，才是这段区间的分配增量；它也不能被称作“只有KV cache”。CUDA上下文和其他库的直接分配未必全部计入。[5]

下面的整数是**教学夹具**，不是显卡测量。

In [5]:
MiB = 1024 * 1024
memory = memory_numbers(100*MiB, 110*MiB, 180*MiB, 256*MiB)
print("区间峰值增量 MiB：", memory["peak_increment_bytes"] / MiB)
print("结束时reserved MiB：", memory["end_reserved_bytes"] / MiB)

区间峰值增量 MiB： 80.0
结束时reserved MiB： 256.0


## 3. 第29课的题集必须原样保留

4道开发题用于确认格式和环境；24道固定评测题分成整数计算、代码阅读、规则分类和JSON提取四类。

我们复用完整题目、system提示、输出要求和严格评分规则。模型请求不含参考答案。正式比较在本课**重新跑FP16基线**，不要直接拿第29课旧总分作对照，因为依赖版本、模板、预算或设备可能已经改变。

本课真实生成固定batch=1、单次贪心、64个新增token、关闭thinking、使用全部仓库EOS。贪心用于控制变量，不是在声称它一定是该模型最佳质量配置。[6]

In [6]:
# %% 02. 原样内置第29课题集与请求协议，无外部文件依赖


def make_suite() -> list[dict]:
    """人工编写的可审查题集；不是公开排行榜、秘密测试集或未知污染的保证。"""
    suite = []

    def add(task, split, number, prompt, expected):
        suite.append({"id": f"{split}-{task}-{number:02d}", "task": task,
                      "split": split, "prompt": prompt, "expected": expected})

    add("arithmetic", "dev", 1, "计算：6 + 7。", 13)
    arithmetic = [
        ("计算：17 + 28。", 45),
        ("计算：-7 + 3。", -4),
        ("计算：12 × 13。", 156),
        ("计算：144 ÷ 12。", 12),
        ("计算：3 × (8 + 2) - 7。", 23),
        ("每支笔6元，购买9支，支付100元，应找回多少元？", 46),
    ]
    for i, (prompt, gold) in enumerate(arithmetic, 1):
        add("arithmetic", "eval", i, prompt, gold)
    add("code_trace", "dev", 1, 'x = [1, 3]\nprint(sum(x))', 4)
    traces = [
        ('a = [2, 4, 6]\nprint(sum(a))', 12),
        ('print(len("模型🙂"))', 3),
        ('total = 0\nfor i in range(5):\n    total += i\nprint(total)', 10),
        ('x = [1, 2]\ny = x\ny.append(3)\nprint(len(x))', 3),
        ('d = {"a": 2, "b": 3}\nprint(d.get("c", 4) + d["a"])', 6),
        ('print(sum(i * i for i in range(4)))', 14),
    ]
    for i, (prompt, gold) in enumerate(traces, 1):
        add("code_trace", "eval", i, prompt, gold)
    add("routing", "dev", 1, "请查看快递状态。", "DELIVERY")
    routes = [
        ("我想退款。", "PAYMENT"),
        ("我的快递到哪了？", "DELIVERY"),
        ("谢谢你的帮助。", "OTHER"),
        ("快递损坏，我要求退款。", "PAYMENT"),
        ("设备已修好，无需维修。", "OTHER"),
        ("我不要求退款，只想咨询。", "PAYMENT"),
    ]
    for i, (prompt, gold) in enumerate(routes, 1):
        add("routing", "eval", i, prompt, gold)
    add("extraction", "dev", 1, "名称：白纸；数量：8。", {"item": "白纸", "quantity": 8})
    extracts = [("红茶", 3), ("A-7", 0), ("螺丝", 12), ("短 USB 线", 2),
                ("3号箱", 1), ('"测试"标签', 4)]
    for i, (name, qty) in enumerate(extracts, 1):
        add("extraction", "eval", i, f"名称：{name}；数量：{qty}。", {"item": name, "quantity": qty})
    validate_suite(suite)
    return suite

def valid_expected(task: str, value: Any) -> bool:
    if task in ("arithmetic", "code_trace"):
        return type(value) is int
    if task == "routing":
        return type(value) is str and value in {"PAYMENT", "DELIVERY", "OTHER"}
    if task == "extraction":
        return (type(value) is dict and set(value) == {"item", "quantity"}
                and type(value["item"]) is str and type(value["quantity"]) is int
                and value["quantity"] >= 0)
    return False

def validate_suite(suite: Any) -> None:
    if type(suite) is not list or not suite:
        raise ValueError("题集必须是非空列表")
    ids, texts = set(), set()
    for item in suite:
        if type(item) is not dict or set(item) != {"id", "task", "split", "prompt", "expected"}:
            raise ValueError("题目字段不符合约定")
        if not isinstance(item["id"], str) or not item["id"] or item["id"] in ids:
            raise ValueError("题目 ID 为空或重复")
        if item["task"] not in TASKS or item["split"] not in {"dev", "eval"}:
            raise ValueError("未知任务或 split")
        if not isinstance(item["prompt"], str) or not item["prompt"].strip():
            raise ValueError("题干必须为非空字符串")
        key = (item["task"], item["prompt"])
        if key in texts:
            raise ValueError("跨 split 或同 split 出现重复题干")
        if not valid_expected(item["task"], item["expected"]):
            raise ValueError("参考答案类型或结构错误")
        ids.add(item["id"])
        texts.add(key)

def select_items(suite: list[dict], split="eval", limit: int | None = None) -> list[dict]:
    validate_suite(suite)
    if split not in {"dev", "eval"}:
        raise ValueError("split 必须是 dev/eval")
    selected = [copy.deepcopy(x) for x in suite if x["split"] == split]
    if not selected:
        raise ValueError("这个 split 为空")
    if limit is not None:
        positive_int(limit, "limit")
        selected = selected[:limit]
    return selected

def public_request(item: dict) -> dict:
    """模型后端只接收这份投影：参考答案和评分器内部状态不进入 messages。"""
    return {"id": item["id"], "messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": CONTRACTS[item["task"]] + "\n\n题目：\n" + item["prompt"]},
    ]}

def protocol_for(items: list[dict]) -> dict:
    validate_suite(items)
    core = {"dataset_sha256": fingerprint(items), "ids": [x["id"] for x in items],
            "request_sha256": fingerprint([public_request(x) for x in items]),
            "grader": GRADER_VERSION,
            "primary_rule": "generation_ok AND valid_format AND correct_value AND stop=eos",
            "normalization": "outer whitespace only; JSON key order/whitespace ignored; strict types"}
    return {**core, "protocol_sha256": fingerprint(core)}

In [7]:
# %% 03. 原样内置第29课评分器：失败/缺失不从分母消失


def strict_json_loads(text: str) -> Any:
    if not isinstance(text, str) or len(text) > 100_000:
        raise ValueError("JSON 必须是长度受限的字符串")

    def pairs_to_dict(pairs):
        result = {}
        for key, value in pairs:
            if key in result:
                raise ValueError(f"JSON 重复键：{key}")
            result[key] = value
        return result

    def reject_constant(value):
        raise ValueError(f"不接受非标准 JSON 常量：{value}")

    return json.loads(text, object_pairs_hook=pairs_to_dict, parse_constant=reject_constant)

def same_typed_value(actual: Any, expected: Any) -> bool:
    """避免 Python 中 True == 1、1.0 == 1 导致宽松评分。"""
    if type(actual) is not type(expected):
        return False
    if type(expected) is dict:
        return (actual.keys() == expected.keys()
                and all(same_typed_value(actual[k], expected[k]) for k in expected))
    if type(expected) is list:
        return len(actual) == len(expected) and all(same_typed_value(a, b) for a, b in zip(actual, expected))
    return actual == expected

def parse_answer(task: str, raw_text: str) -> tuple[Any, str | None]:
    if not isinstance(raw_text, str):
        return None, "output_not_string"
    if len(raw_text) > 100_000:
        return None, "output_too_long"
    text = raw_text.strip()  # 不删除内部空格、标点、大小写、推理过程或代码围栏。
    if not text:
        return None, "empty_output"
    if task in {"arithmetic", "code_trace"}:
        if len(text) > 1000 or re.fullmatch(r"(?:0|-[1-9][0-9]*|[1-9][0-9]*)", text) is None:
            return None, "integer_format"
        return int(text), None
    if task == "routing":
        if text not in {"PAYMENT", "DELIVERY", "OTHER"}:
            return None, "label_format"
        return text, None
    if task == "extraction":
        try:
            value = strict_json_loads(text)
        except (ValueError, RecursionError):
            return None, "json_syntax_or_duplicate_key"
        if not valid_expected(task, value):
            return value, "json_schema"
        return value, None
    raise ValueError("未知任务")

def validate_raw_record(row: Any) -> None:
    if not isinstance(row, dict) or not isinstance(row.get("id"), str):
        raise ValueError("响应必须包含字符串 id")
    if row.get("status") not in {"ok", "error"}:
        raise ValueError("status 必须是 ok/error")
    if row["status"] == "ok":
        if not isinstance(row.get("raw_text"), str):
            raise ValueError("ok 响应缺少 raw_text")
        if row.get("stop_reason") not in {"eos", "length", "other"}:
            raise ValueError("未知停止原因")
    elif not isinstance(row.get("error"), dict):
        raise ValueError("error 响应必须包含错误详情")

def grade_one(item: dict, record: dict | None) -> dict:
    result = {"id": item["id"], "task": item["task"], "present": record is not None,
              "generated": False, "format_valid": False, "answer_correct": False,
              "success": False, "reason": "missing", "parsed": None}
    if record is None:
        return result
    validate_raw_record(record)
    if record["id"] != item["id"]:
        raise ValueError("题目和响应 ID 不一致")
    if record["status"] == "error":
        return {**result, "reason": "generation_error"}
    parsed, error = parse_answer(item["task"], record["raw_text"])
    result.update(generated=True, parsed=parsed, format_valid=error is None)
    if error is not None:
        return {**result, "reason": error}
    correct = same_typed_value(parsed, item["expected"])
    result["answer_correct"] = correct
    if not correct:
        return {**result, "reason": "wrong_value"}
    if record["stop_reason"] != "eos":
        return {**result, "reason": "correct_but_not_eos"}
    return {**result, "success": True, "reason": "success"}

def summary_metrics(graded: list[dict], records: list[dict]) -> dict:
    if not graded:
        raise ValueError("不能汇总空题集")
    n = len(graded)
    by_task = {}
    for task in sorted({x["task"] for x in graded}):
        subset = [x for x in graded if x["task"] == task]
        by_task[task] = {"n": len(subset), "success": sum(x["success"] for x in subset),
                         "success_rate": sum(x["success"] for x in subset) / len(subset)}
    ok = [x for x in records if x["status"] == "ok"]
    durations = [x["generate_seconds"] for x in ok
                 if type(x.get("generate_seconds")) in (int, float)
                 and math.isfinite(x["generate_seconds"]) and x["generate_seconds"] >= 0]
    token_counts = [x["generated_tokens"] for x in ok if type(x.get("generated_tokens")) is int]
    result = {"n": n, "received": sum(x["present"] for x in graded),
              "generated": sum(x["generated"] for x in graded),
              "format_valid": sum(x["format_valid"] for x in graded),
              "answer_correct": sum(x["answer_correct"] for x in graded),
              "success": sum(x["success"] for x in graded),
              "micro_success_rate": sum(x["success"] for x in graded) / n,
              "macro_success_rate": statistics.mean(x["success_rate"] for x in by_task.values()),
              "generation_coverage": sum(x["generated"] for x in graded) / n,
              "answer_accuracy": sum(x["answer_correct"] for x in graded) / n,
              "format_rate_all_items": sum(x["format_valid"] for x in graded) / n,
              "per_task": by_task, "failure_counts": dict(Counter(x["reason"] for x in graded)),
              "stop_counts_generated_only": dict(Counter(x["stop_reason"] for x in ok)),
              "median_generate_seconds": statistics.median(durations) if durations else None,
              "mean_generated_tokens": statistics.mean(token_counts) if token_counts else None,
              "timing_samples": len(durations),
              "timing_scope": "single-request generate, prefill+decode; no load/tokenization; not TTFT or controlled benchmark"}
    return result

def score_records(items: list[dict], records: list[dict]) -> dict:
    validate_suite(items)
    expected_ids = {x["id"] for x in items}
    indexed = {}
    for row in records:
        validate_raw_record(row)
        if row["id"] not in expected_ids:
            raise ValueError("响应包含题集之外的 ID")
        if row["id"] in indexed:
            raise ValueError("重复响应 ID；禁止重试后挑选最好的回答")
        indexed[row["id"]] = row
    graded = [grade_one(item, indexed.get(item["id"])) for item in items]
    return {"graded": graded, "metrics": summary_metrics(graded, records)}

def build_report(items: list[dict], records: list[dict], *, kind: str,
                 condition: dict, scope: str, complete: bool = True) -> dict:
    if kind not in {"hand_authored_fixture", "real_model"}:
        raise ValueError("必须明确是真实模型还是手工夹具")
    if scope not in {"dev", "full_eval", "smoke_subset"}:
        raise ValueError("未知评测范围")
    scored = score_records(items, records)
    return {"format": FORMAT_VERSION, "kind": kind, "scope": scope, "complete": complete,
            "protocol": protocol_for(items), "condition": copy.deepcopy(condition),
            "items": copy.deepcopy(items), "raw_records": copy.deepcopy(records),
            "raw_records_sha256": fingerprint(records), **scored,
            "limitations": ["人工教学题集，不是通用能力基准；不能证明训练污染不存在。",
                            "严格成功率要求正确且正常EOS结束，是本课协议，不是所有基准的通用规则。",
                            "看过评测答案后调参，题集就成为开发/回归集；不可再宣称独立盲测。"]}

def verify_report(report: dict) -> dict:
    if report.get("format") != FORMAT_VERSION:
        raise ValueError("报告格式不匹配")
    if report.get("protocol") != protocol_for(report["items"]):
        raise ValueError("题集/提示/评分协议指纹不匹配")
    if report.get("raw_records_sha256") != fingerprint(report["raw_records"]):
        raise ValueError("原始响应指纹不匹配")
    fresh = score_records(report["items"], report["raw_records"])
    if fresh["graded"] != report.get("graded") or fresh["metrics"] != report.get("metrics"):
        raise ValueError("保存的评分与原始响应不一致；必须重新评分")
    return fresh

def paired_compare(before: dict, after: dict) -> dict:
    verify_report(before)
    verify_report(after)
    if not before["complete"] or not after["complete"]:
        raise ValueError("未完成的运行不能宣称完整对比")
    if before["kind"] != after["kind"]:
        raise ValueError("不能把手工夹具与真实模型成绩比较")
    if before["protocol"] != after["protocol"] or before["scope"] != after["scope"]:
        raise ValueError("配对比较要求相同题集、提示、评分器、范围与题目ID顺序")
    a = {x["id"]: x["success"] for x in before["graded"]}
    b = {x["id"]: x["success"] for x in after["graded"]}
    fixed = [k for k in a if not a[k] and b[k]]
    regressed = [k for k in a if a[k] and not b[k]]
    changed = {k: {"before": before["condition"].get(k), "after": after["condition"].get(k)}
               for k in sorted(before["condition"].keys() | after["condition"].keys())
               if before["condition"].get(k) != after["condition"].get(k)}
    return {"n": len(a), "before_success": sum(a.values()), "after_success": sum(b.values()),
            "fixed_ids": fixed, "regressed_ids": regressed,
            "both_correct": sum(a[k] and b[k] for k in a),
            "both_incorrect": sum(not a[k] and not b[k] for k in a),
            "delta_percentage_points": 100 * (sum(b.values()) - sum(a.values())) / len(a),
            "changed_conditions": changed,
            "note": "配对描述而非显著性检验；多项条件改变时不得归因于单一改动。"}

def stable_item_seed(base: int, item_id: str) -> int:
    if type(base) is not int or base < 0:
        raise ValueError("seed 必须是非负整数")
    # 不使用 Python hash()，其进程间结果可能改变。
    return int(fingerprint({"seed": base, "id": item_id})[:8], 16) % (2**31)

def split_generated_ids(sequence: list[int], prefix: list[int], eos: list[int], budget: int) -> dict:
    positive_int(budget, "budget")
    if sequence[:len(prefix)] != prefix or len(sequence) < len(prefix):
        raise ValueError("生成结果没有保留原输入前缀")
    new = sequence[len(prefix):]
    if len(new) > budget:
        raise ValueError("生成长度超过设定预算")
    positions = [i for i, token in enumerate(new) if token in eos]
    if positions:
        first = positions[0]
        return {"visible_ids": new[:first], "generated_ids": new[:first + 1], "stop_reason": "eos"}
    return {"visible_ids": new, "generated_ids": new,
            "stop_reason": "length" if len(new) == budget else "other"}

EXPECTED_SUITE_SHA256 = '5531e2a27294d8807f43dbedbdd4de054db1c9430a930995667b9fc936870961'

EXPECTED_EVAL_PROTOCOL_SHA256 = '61697eb46f8d5d1cfb464e37e91eab27cf455123ca4ab4df184dea59de3f5350'

In [8]:
suite = make_suite()
items = select_items(suite, "eval")
print("开发题：", len(select_items(suite, "dev")), "评测题：", len(items))
print("完整题集指纹：", fingerprint(suite))
print("评测协议指纹：", protocol_for(items)["protocol_sha256"])
print("第一道公开请求：", json.dumps(public_request(items[0]), ensure_ascii=False, indent=2))
assert fingerprint(suite) == EXPECTED_SUITE_SHA256
assert protocol_for(items)["protocol_sha256"] == EXPECTED_EVAL_PROTOCOL_SHA256

开发题： 4 评测题： 24
完整题集指纹： 5531e2a27294d8807f43dbedbdd4de054db1c9430a930995667b9fc936870961
评测协议指纹： 61697eb46f8d5d1cfb464e37e91eab27cf455123ca4ab4df184dea59de3f5350
第一道公开请求： {
  "id": "eval-arithmetic-01",
  "messages": [
    {
      "role": "system",
      "content": "你正在完成固定题集。请严格遵守每道题的输出格式，只输出最终答案，不解释、不使用 Markdown 代码围栏。"
    },
    {
      "role": "user",
      "content": "只输出一个十进制整数。使用 ASCII 数字和必要的负号，不加单位、前导零、加号或解释。\n\n题目：\n计算：17 + 28。"
    }
  ]
}


### 3.1 同分数，仍可能有回归

下面人为准备两份记录：第一份错一道，第二份修好它、同时错了另一道。两份都是23/24，但不是相同行为。

这里的响应由参考答案手工构造，仅测试评分程序，绝不是模型回答。

In [9]:
# %% 12. 离线教学夹具与练习参考实现：绝不冒充真实模型成绩

def reference_success_records(items: list[dict]) -> list[dict]:
    records = []
    for item in items:
        value = item["expected"]
        text = json.dumps(value, ensure_ascii=False) if type(value) is dict else str(value)
        records.append({"id": item["id"], "status": "ok", "raw_text": text, "stop_reason": "eos"})
    return records


def regression_fixture() -> dict:
    items = select_items(make_suite(), "eval")
    a = reference_success_records(items)
    a[0]["raw_text"] = "0"  # 45 -> wrong
    b = copy.deepcopy(a)
    b[0]["raw_text"] = "45" # fix one
    b[1]["raw_text"] = "4"  # -4 -> wrong
    args = {"kind": "hand_authored_fixture", "scope": "full_eval", "condition": {"origin": "manual"}}
    before = build_report(items, a, **args)
    after = build_report(items, b, **args)
    paired = paired_compare(before, after)
    return {"kind": "hand_authored_fixture_not_model_result", "before": before, "after": after, "paired": paired}


def checkpoint_failure_example() -> dict:
    items = select_items(make_suite(), "eval")
    raw = reference_success_records(items)
    raw[0] = {"id": items[0]["id"], "status": "error", "error": {"type":"ExampleOOM", "message":"hand-authored, no actual OOM"}}
    raw.pop()
    return score_records(items, raw)


def offline_demo() -> dict:
    fixture = regression_fixture()
    return {"real_model_run": False,
            "suite_sha256": fingerprint(make_suite()),
            "eval_protocol_sha256": protocol_for(select_items(make_suite(), "eval"))["protocol_sha256"],
            "plans": {mode: quantization_plan(mode) for mode in MODES},
            "analytical_storage": ideal_storage_demo(),
            "manual_fixture_comparison": fixture["paired"],
            "missing_and_error_fixture": checkpoint_failure_example()["metrics"],
            "note": "No model, VRAM or throughput results are produced in offline mode."}


def quant_config_answer(mode, torch_module, config_factory):
    return make_quant_config(mode, torch_module, config_factory)


def memory_answer(start, live, peak, reserved):
    return memory_numbers(start, live, peak, reserved)


def paired_success_answer(items: list[dict], a: list[dict], b: list[dict]) -> dict:
    # 复用严格评分器，但自己按ID对齐，不只看总分。
    aa = score_records(items, a)
    bb = score_records(items, b)
    indexed = {r["id"]: r for r in bb["graded"]}
    fixed, regressed = [], []
    for row in aa["graded"]:
        later = indexed[row["id"]]
        if not row["success"] and later["success"]:
            fixed.append(row["id"])
        if row["success"] and not later["success"]:
            regressed.append(row["id"])
    return {"n": len(items), "before": aa["metrics"]["success"], "after": bb["metrics"]["success"],
            "fixed_ids": fixed, "regressed_ids": regressed}

In [10]:
fixture = regression_fixture()
print(json.dumps(fixture["paired"], ensure_ascii=False, indent=2))

{
  "n": 24,
  "before_success": 23,
  "after_success": 23,
  "fixed_ids": [
    "eval-arithmetic-01"
  ],
  "regressed_ids": [
    "eval-arithmetic-02"
  ],
  "both_correct": 22,
  "both_incorrect": 0,
  "delta_percentage_points": 0.0,
  "changed_conditions": {},
  "note": "配对描述而非显著性检验；多项条件改变时不得归因于单一改动。"
}


### 3.2 运行失败与质量退步不能混为一谈

对单个题集，运行错误或缺失题目仍保留在计划分母里，避免只报成功题。另一方面，若一个模式根本没加载成功，就不应把它作为“量化后模型0分”的能力结论。

本课为每个模式保存 `status` 和失败阶段，未完成模式不参加成功模型的配对排名。异常时不偷偷缩小模型、回退CPU、缩短生成预算或反复重试挑最好答案。

In [11]:
failed_fixture = checkpoint_failure_example()
print({k: failed_fixture["metrics"][k] for k in ("n", "received", "generated", "success")})
print("夹具中1题error、1题missing，分母仍为24。")

{'n': 24, 'received': 23, 'generated': 22, 'success': 22}
夹具中1题error、1题missing，分母仍为24。


## 4. 环境与安装：先看运行报告，不凭显卡名字猜成功

RTX 3060属于本课面向的NVIDIA单卡路径；具体还要看PyTorch CUDA构建、驱动、bitsandbytes轮子和系统。官方安装指南列出了不同特性的硬件要求。[7]

在**已有可用CUDA PyTorch的环境**中安装配套依赖：

```bash
python -m pip install "transformers>=4.57.3,<5" "accelerate>=1.2,<2" "bitsandbytes>=0.48,<0.51" safetensors
```

这是本课采用的接口版本范围，不是“最新版本推荐”，更不是本制作环境已验证的CUDA兼容矩阵。已有5.x环境时，建议另建课程环境，别在重要项目里直接降级。

本课不提供一个通用的 `pip install torch` 来覆盖你已经可用的CUDA环境，也不需要另装 `flash-attn`。

如果Jupyter找不到已装的库，先确认内核的 `sys.executable`。所有真实版本会写入报告。

In [12]:
# %% 05. 环境检查，不联网、不安装依赖

def doctor(gpu: int = 0) -> dict:
    versions = {}
    for package in ("torch", "transformers", "accelerate", "bitsandbytes", "safetensors", "tokenizers", "huggingface_hub"):
        try:
            versions[package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            versions[package] = None
    report = {"python": platform.python_version(), "platform": platform.platform(),
              "packages": versions, "cuda_available": False, "gpu": None,
              "bnb_import_error": None, "real_model_loaded": False}
    try:
        import torch
        report.update(torch_cuda_build=torch.version.cuda, cuda_available=torch.cuda.is_available())
        if torch.cuda.is_available() and 0 <= gpu < torch.cuda.device_count():
            prop = torch.cuda.get_device_properties(gpu)
            report["gpu"] = {"index": gpu, "name": prop.name,
                             "compute_capability": list(torch.cuda.get_device_capability(gpu)),
                             "total_memory_bytes": prop.total_memory}
    except ImportError as error:
        report["torch_error"] = str(error)
    if versions["bitsandbytes"]:
        try:
            import bitsandbytes  # noqa: F401 — 捕获二进制导入失败，不把pip存在当作可运行
        except Exception as error:
            report["bnb_import_error"] = f"{type(error).__name__}: {error}"
    return report


def require_runtime(mode: str | None, gpu: int, require_cuda: bool = True):
    try:
        import torch
        import transformers
        from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM, GenerationConfig, BitsAndBytesConfig
    except ImportError as error:
        raise RuntimeError("真实路径需要现有PyTorch，以及 transformers>=4.57.3,<5、accelerate、safetensors；量化还需bitsandbytes。") from error
    version = tuple(int(x) for x in re.findall(r"\d+", transformers.__version__)[:3])
    if not (4, 57, 3) <= version < (5, 0, 0):
        raise RuntimeError("本课固定4.57系列接口范围>=4.57.3,<5，不把未验证的5.x改动静默混入对照。")
    if require_cuda:
        if not torch.cuda.is_available() or not 0 <= gpu < torch.cuda.device_count():
            raise RuntimeError("本课真实三路对照限定单张CUDA卡；当前不可用，不回退CPU。离线实验可继续。")
        torch.cuda.set_device(gpu)
        if mode in {"int8", "nf4"}:
            try:
                import bitsandbytes  # noqa: F401
            except Exception as error:
                raise RuntimeError("bitsandbytes二进制加载失败；先检查CUDA/PyTorch/bnb版本匹配。") from error
    return torch, AutoConfig, AutoTokenizer, AutoModelForCausalLM, GenerationConfig, BitsAndBytesConfig

In [13]:
print("当前内核：", sys.executable)
ENVIRONMENT = doctor(0)
print(json.dumps(ENVIRONMENT, ensure_ascii=False, indent=2))
print("这只是环境检查，尚未加载权重。")

当前内核： /opt/pyvenv/bin/python


{
  "python": "3.13.5",
  "platform": "Linux-6.18.57-x86_64-with-glibc2.41",
  "packages": {
    "torch": "2.10.0+cpu",
    "transformers": null,
    "accelerate": null,
    "bitsandbytes": null,
    "safetensors": "0.7.0",
    "tokenizers": null,
    "huggingface_hub": "1.16.1"
  },
  "cuda_available": false,
  "gpu": null,
  "bnb_import_error": null,
  "real_model_loaded": false,
  "torch_cuda_build": null
}
这只是环境检查，尚未加载权重。


## 5. 固定输入合同：模型来源、模板、编号和生成规则一起锁定

除了第29课的题目协议，本课还比较模型commit、模板指纹、分词器指纹、实际输入ID、EOS、生成预算、软件版本、GPU和注意力实现。

**有意改变的只有加载方案。** INT8与NF4的数值算法不同，所以这仍是“加载方案对比”，不是把所有因素归结为位宽。

下面定义真实检查函数；定义本身不联网。真正下载仅发生在后面明确开启的单元中。首次加载仍可能下载原始高精度权重，再在加载过程中量化；不是自动下载一个小得多的专用四位文件。[1]

In [14]:
# %% 06. 固定模型版本、配套模板和完全相同的输入编号

def inspect_inputs(s: Settings) -> tuple:
    validate_settings(s)
    torch, AutoConfig, AutoTokenizer, AutoModel, GenConfig, BnbConfig = require_runtime(None, s.gpu, False)
    kwargs = {"revision": REVISION, "local_files_only": s.local_files_only, "trust_remote_code": False}
    config = AutoConfig.from_pretrained(MODEL_ID, **kwargs)
    if config.model_type != "qwen3":
        raise ValueError("本课仅接受固定版本Qwen3")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, padding_side="left", **kwargs)
    if not isinstance(tokenizer.chat_template, str) or "enable_thinking" not in tokenizer.chat_template:
        raise ValueError("缺少所需的Qwen3聊天模板")
    if tokenizer.pad_token_id is None or max(tokenizer.get_vocab().values()) >= config.vocab_size:
        raise ValueError("配套tokenizer的PAD/词表范围不符合配置")
    gen = GenConfig.from_pretrained(MODEL_ID, revision=REVISION, local_files_only=s.local_files_only)
    eos = gen.eos_token_id
    eos = [eos] if type(eos) is int else list(eos or [])
    options = generation_options(s, eos, tokenizer.pad_token_id)
    items = select_items(make_suite(), s.split)
    requests = []
    for item in items:
        request = public_request(item)
        rendered = tokenizer.apply_chat_template(request["messages"], tokenize=False,
                                                  add_generation_prompt=True, enable_thinking=False)
        ids = tokenizer(rendered, add_special_tokens=False, truncation=False)["input_ids"]
        direct = tokenizer.apply_chat_template(request["messages"], tokenize=True,
                                               add_generation_prompt=True, enable_thinking=False)
        if ids != direct:
            raise ValueError("模板两条编码路线不一致")
        if not ids or len(ids) > s.max_input or len(ids) + s.max_new > min(s.context_limit, config.max_position_embeddings):
            raise ValueError(f"{item['id']} 超过预算；不会静默截断")
        requests.append({**request, "input_ids": ids, "rendered_prompt": rendered})
    # 清除不影响数学结构但可能因本地缓存路径变化的字段。
    config_dict = config.to_dict()
    config_dict.pop("_name_or_path", None)
    info = {"model": MODEL_ID, "revision": REVISION,
            "config_sha256": fingerprint(config_dict),
            "template_sha256": fingerprint(tokenizer.chat_template),
            "tokenizer_sha256": fingerprint(tokenizer.backend_tokenizer.to_str()),
            "input_ids_sha256": fingerprint([{ "id": r["id"], "input_ids": r["input_ids"]} for r in requests]),
            "generation_options": options, "enable_thinking": False,
            "model_config": config_dict, "requests": requests, "weights_loaded": False}
    return config, tokenizer, items, requests, options, info


def fair_contract(s: Settings, info: dict, env: dict, items: list[dict]) -> dict:
    """仅排除本次有意改变的量化方式；其余关键条件必须相同。"""
    return {"model": info["model"], "revision": info["revision"],
            "config_sha256": info["config_sha256"],
            "template_sha256": info["template_sha256"],
            "tokenizer_sha256": info["tokenizer_sha256"],
            "input_ids_sha256": info["input_ids_sha256"],
            "protocol": protocol_for(items), "generation_options": info["generation_options"],
            "enable_thinking": False, "batch_size": 1, "seed": s.seed,
            "non_quant_dtype": "float16", "attention": "sdpa", "offload": False,
            "cache": "dynamic_unquantized", "gpu": env["gpu"],
            "packages": env["packages"], "python": env["python"],
            "platform": env["platform"], "cuda_build": env.get("torch_cuda_build")}


def require_matching_contract(a: dict, b: dict) -> None:
    if a != b:
        changed = [k for k in sorted(a.keys() | b.keys()) if a.get(k) != b.get(k)]
        raise ValueError("对照条件不一致，拒绝单独归因于量化：" + ", ".join(changed))

In [15]:
contract_a = {"revision": "same", "input_ids": [1, 2], "max_new": 64}
contract_b = {**contract_a, "max_new": 128}
try:
    require_matching_contract(contract_a, contract_b)
except ValueError as error:
    print("预期拒绝：", error)

预期拒绝： 对照条件不一致，拒绝单独归因于量化：max_new


## 6. 验收模块：加载标志不是证据

本课对每层Qwen3的7个投影逐个验收：Q/K/V/O以及MLP的gate/up/down。

| 模式 | 预期模块 | 主要权重存储类型 |
|---|---|---|
| FP16 | 普通 `Linear` | `float16` |
| INT8 | `bitsandbytes.nn.Linear8bitLt` | `int8` |
| NF4 | `bitsandbytes.nn.Linear4bit` | 打包的 `uint8` 容器 |

NF4还要检查量化状态、NF4格式、双重量化和计算dtype；INT8检查没有保留用于训练的FP16主权重。Embedding和`lm_head`保持未量化，记录共享存储情况。

检查发生在一次真实前向之后，使延迟初始化的状态已经建立。检查不通过就停止该分支，不让“能回答一句话”替代量化正确性验收。[1][4]

设备固定为 `device_map={"": gpu_index}`；不采用可能把层放到CPU/disk的自动卸载。加载后不再调用 `.half()`、`.dequantize()` 或对整网重新 `.to()`。

In [16]:
# %% 07. 验收模块类型、实际存储类型、量化状态与设备

def audit_modules(model, mode: str, gpu: int, bnb_module=None) -> dict:
    import torch
    if mode not in MODES:
        raise ValueError("未知模式")
    if bnb_module is None and mode != "fp16":
        import bitsandbytes as bnb_module
    types4 = (bnb_module.nn.Linear4bit,) if bnb_module is not None else ()
    types8 = (bnb_module.nn.Linear8bitLt,) if bnb_module is not None else ()
    rows = []
    for name, layer in model.named_modules():
        if isinstance(layer, types4):
            kind = "Linear4bit"
        elif isinstance(layer, types8):
            kind = "Linear8bitLt"
        elif isinstance(layer, torch.nn.Linear):
            kind = "Linear"
        else:
            continue
        weight = layer.weight
        state = getattr(weight, "quant_state", None)
        row = {"name": name, "kind": kind, "class": f"{type(layer).__module__}.{type(layer).__name__}",
               "weight_class": type(weight).__name__, "weight_dtype": str(weight.dtype),
               "weight_device": str(weight.device), "physical_weight_shape": list(weight.shape),
               "physical_weight_elements": weight.numel(),
               "logical_weight_elements": layer.in_features * layer.out_features,
               "in_features": layer.in_features, "out_features": layer.out_features,
               "compute_dtype": str(getattr(layer, "compute_dtype", None)),
               "has_quant_state": state is not None,
               "quant_type": getattr(state, "quant_type", getattr(weight, "quant_type", None)),
               "nested": bool(getattr(state, "nested", False)),
               "has_fp16_weights": getattr(weight, "has_fp16_weights", None)}
        rows.append(row)
    validate_projection_inventory(rows, mode, model.config.num_hidden_layers)
    devices = {name: str(p.device) for name, p in model.named_parameters()}
    assert_single_gpu_devices(devices, gpu)
    # registered tensors 不一定包含bitsandbytes量化元数据，不能称为完整模型显存。
    dtype_counts = Counter()
    for p in model.parameters():
        dtype_counts[str(p.dtype)] += p.numel()
    embedding = model.get_input_embeddings().weight
    output = model.get_output_embeddings().weight
    return {"mode": mode, "passed": True, "linear_modules": rows,
            "counts": dict(Counter(r["kind"] for r in rows)),
            "registered_parameter_elements_by_storage_dtype": dict(dtype_counts),
            "get_memory_footprint_bytes": int(model.get_memory_footprint()),
            "footprint_scope": "library parameter/buffer accounting; not complete quant metadata or GPU peak",
            "embedding_dtype": str(embedding.dtype), "lm_head_dtype": str(output.dtype),
            "tied_input_output_storage": embedding.data_ptr() == output.data_ptr(),
            "hf_device_map": {k: str(v) for k, v in getattr(model, "hf_device_map", {}).items()},
            "all_parameters_on_requested_gpu": True}


def inspect_cache(cache) -> dict:
    """仅观察真正返回的缓存，不用'4bit模型'推断缓存也是4bit。"""
    import torch
    if cache is None:
        return {"available": False, "reason": "backend did not return cache", "payload_bytes": None}
    pairs = []
    if hasattr(cache, "layers"):
        pairs = [(getattr(layer, "keys", None), getattr(layer, "values", None)) for layer in cache.layers]
    elif hasattr(cache, "key_cache"):
        pairs = list(zip(cache.key_cache, cache.value_cache))
    elif isinstance(cache, (tuple, list)):
        pairs = [(layer[0], layer[1]) for layer in cache if len(layer) >= 2]
    good = [(k, v) for k, v in pairs if isinstance(k, torch.Tensor) and isinstance(v, torch.Tensor)]
    if not good:
        return {"available": False, "reason": "unsupported or empty cache representation", "payload_bytes": None}
    return {"available": True, "class": type(cache).__name__, "layers": len(good),
            "payload_bytes": sum(t.numel() * t.element_size() for pair in good for t in pair),
            "dtypes": sorted({str(t.dtype) for pair in good for t in pair}),
            "first_layer_key_shape": list(good[0][0].shape),
            "note": "effective KV tensor payload, excludes temporary/allocator overhead; last selected token may not yet be cached"}


def cuda_memory_report(torch, device, start: int) -> dict:
    torch.cuda.synchronize(device)
    return memory_numbers(start, torch.cuda.memory_allocated(device),
                          torch.cuda.max_memory_allocated(device), torch.cuda.memory_reserved(device))

In [17]:
print("一个Transformer层需要检查的投影：")
for name in required_projections(1):
    print(" ", name)
print("设备guard演示：")
try:
    assert_single_gpu_devices({"a": "cuda:0", "b": "cpu"}, 0)
except ValueError as error:
    print("预期拒绝：", error)

一个Transformer层需要检查的投影：
  model.layers.0.self_attn.q_proj
  model.layers.0.self_attn.k_proj
  model.layers.0.self_attn.v_proj
  model.layers.0.self_attn.o_proj
  model.layers.0.mlp.gate_proj
  model.layers.0.mlp.up_proj
  model.layers.0.mlp.down_proj
设备guard演示：
预期拒绝： 本课禁止CPU/disk/meta/其他GPU回退：{'b': 'cpu'}


### 6.1 权重量化不会自动量化KV cache

本课明确使用普通dynamic cache，不启用cache quantization。报告从真实返回的K/V张量读取dtype、形状和元素载荷。

它可能仍是FP16；上下文增长时，KV占用仍会增加。因此“权重放得下”与“所需上下文放得下”是两个问题。

下面只用CPU张量验证缓存统计接口，数字不是Qwen缓存实测。

In [18]:
try:
    import torch
    from types import SimpleNamespace
    fake_cache = SimpleNamespace(layers=[SimpleNamespace(
        keys=torch.zeros(1, 2, 3, 4, dtype=torch.float16),
        values=torch.zeros(1, 2, 3, 4, dtype=torch.float16),
    )])
    print("教学张量：", inspect_cache(fake_cache))
except ImportError:
    print("未安装PyTorch，跳过张量演示。")

教学张量： {'available': True, 'class': 'SimpleNamespace', 'layers': 1, 'payload_bytes': 96, 'dtypes': ['torch.float16'], 'first_layer_key_shape': [1, 2, 3, 4], 'note': 'effective KV tensor payload, excludes temporary/allocator overhead; last selected token may not yet be cached'}


## 7. 固定前缀诊断：不要比较已经分叉的两段续写

相同提示经过不同加载方案后，最后位置的原始logits可能发生变化。可以比较最大绝对误差、RMSE、概率分布KL和top-1是否一致。

但量化分支一旦选出不同的第一个token，后续输入本身就不同了。此时把第二步logits差异全部叫作“量化数值误差”，会混入上下文差异。

本课只对同一固定提示保存一份末位置logits。它是数值诊断，不是整套任务的能力评分。

In [19]:
# %% 08. 每题独立推理；诊断只比较完全相同前缀的logits

def generate_request(model, tokenizer, request: dict, options: dict, s: Settings) -> dict:
    import torch
    if "expected" in request or set(request) != {"id", "messages", "input_ids", "rendered_prompt"}:
        raise ValueError("推理侧只能收到请求，不能收到参考答案")
    device = torch.device(f"cuda:{s.gpu}")
    ids = torch.tensor([request["input_ids"]], dtype=torch.long, device=device)
    mask = torch.ones_like(ids)
    if ids.shape[1] > s.max_input or ids.shape[1] + s.max_new > s.context_limit:
        raise ValueError("生成预算超限")
    from transformers import GenerationConfig
    cfg = GenerationConfig(**options)
    seed = stable_item_seed(s.seed, request["id"])
    model.eval()
    with torch.random.fork_rng(devices=[s.gpu]), torch.inference_mode():
        torch.manual_seed(seed)
        torch.cuda.synchronize(device)
        start_alloc = torch.cuda.memory_allocated(device)
        torch.cuda.reset_peak_memory_stats(device)
        start = time.perf_counter()
        out = model.generate(input_ids=ids, attention_mask=mask, generation_config=cfg)
        torch.cuda.synchronize(device)
        seconds = time.perf_counter() - start
        memory = cuda_memory_report(torch, device, start_alloc)
        result = split_generated_ids(out.sequences[0].cpu().tolist(), request["input_ids"],
                                     options["eos_token_id"], s.max_new)
        cache = inspect_cache(getattr(out, "past_key_values", None))
        text = tokenizer.decode(result["visible_ids"], skip_special_tokens=False,
                                clean_up_tokenization_spaces=False)
    record = {"id": request["id"], "status": "ok", "raw_text": text,
              "stop_reason": result["stop_reason"], "input_tokens": len(request["input_ids"]),
              "generated_tokens": len(result["generated_ids"]),
              "input_ids": request["input_ids"], "generated_token_ids": result["generated_ids"],
              "effective_seed": seed, "generation_config": options,
              "generate_seconds": seconds, "memory": memory, "cache": cache}
    del out, ids, mask
    return record


def fixed_prefix_probe(model, request: dict, gpu: int, target: Path) -> dict:
    import torch
    ids = torch.tensor([request["input_ids"]], dtype=torch.long, device=f"cuda:{gpu}")
    with torch.inference_mode():
        # 固定的Qwen3 4.57接口支持logits_to_keep=1，避免返回全部位置的词表矩阵。
        out = model(input_ids=ids, attention_mask=torch.ones_like(ids), use_cache=False,
                    return_dict=True, logits_to_keep=1)
        logits = out.logits[0, -1].float().cpu().contiguous()
    if not torch.isfinite(logits).all():
        raise FloatingPointError("固定前缀logits含非有限值")
    torch.save({"request_id": request["id"], "input_ids": request["input_ids"], "logits": logits}, target)
    top = torch.topk(logits, 5)
    result = {"request_id": request["id"], "input_ids_sha256": fingerprint(request["input_ids"]),
              "logits_shape": list(logits.shape), "finite": True,
              "top5_ids": top.indices.tolist(), "top5_logits": top.values.tolist(),
              "scope": "same fixed prompt, last-position raw logits; not a capability metric"}
    del out, ids, logits
    return result


def logits_difference(a, b) -> dict:
    import torch
    if a.shape != b.shape or a.ndim != 1 or a.numel() < 2:
        raise ValueError("必须比较同形状的一维词表logits")
    a, b = a.double(), b.double()
    if not torch.isfinite(a).all() or not torch.isfinite(b).all():
        raise ValueError("logits必须有限")
    diff = b - a
    pa, pb = torch.log_softmax(a, -1), torch.log_softmax(b, -1)
    return {"max_abs": diff.abs().max().item(), "mean_abs": diff.abs().mean().item(),
            "rmse": diff.square().mean().sqrt().item(),
            "kl_fp16_to_variant": max(0.0, (pa.exp() * (pa - pb)).sum().item()),
            "baseline_top1": a.argmax().item(), "variant_top1": b.argmax().item(),
            "top1_same": bool(a.argmax() == b.argmax())}

In [20]:
try:
    import torch
    a = torch.tensor([0.501, 0.499])
    b = torch.tensor([0.499, 0.501])
    print("教学分数，不是模型输出：", logits_difference(a, b))
except ImportError:
    print("未安装PyTorch，跳过logits演示。")

教学分数，不是模型输出： {'max_abs': 0.001999974250793457, 'mean_abs': 0.001999974250793457, 'rmse': 0.001999974250793457, 'kl_fp16_to_variant': 1.999947835286094e-06, 'baseline_top1': 0, 'variant_top1': 1, 'top1_same': False}


## 8. 评测耗时与固定工作量基准必须分开

24题的输出长度可能因模型而异，甚至某些回答提前EOS。不能看到总耗时更低就断言量化内核更快。

可选基准固定同一提示与后续token轨迹：先prefill一次，再依次喂入预先确定的token。默认共产生16次预测，即一次prefill和15次decode，不通过EOS提前结束。

它测的是完整模型前向、动态KV追加与末位置输出头；不含下载、分词、采样、服务排队。每模式预热2次，重复5次，计时边界同步CUDA，保留原始样本和中位数。[5]

**它不是线上TTFT或用户体验延迟**，也不证明“位数越少越快”。小模型、短序列下，量化额外处理可能影响收益。各模式虽然是新进程，仍应重复测量并改变顺序，排查温度、时钟和外部负载影响。

In [21]:
# %% 09. 可选固定工作量基准：与题目生成耗时分开

def benchmark_fixed_tokens(model, tokenizer, request: dict, s: Settings) -> dict:
    import torch
    device = torch.device(f"cuda:{s.gpu}")
    prompt = torch.tensor([request["input_ids"]], device=device, dtype=torch.long)
    continuation = tokenizer.encode("学习模型。", add_special_tokens=False)
    if not continuation:
        raise ValueError("基准续接token为空")
    n = s.benchmark_steps
    if prompt.shape[1] + n - 1 > s.context_limit:
        raise ValueError("基准总长度超过上下文预算")
    token_tensors = [torch.tensor([[continuation[i % len(continuation)]]], device=device)
                     for i in range(n - 1)]
    masks = [torch.ones((1, prompt.shape[1] + i + 1), dtype=torch.long, device=device)
             for i in range(n - 1)]

    def one():
        # 不选择模型生成的token；各模式喂入完全相同的固定token轨迹。
        torch.cuda.synchronize(device)
        start_alloc = torch.cuda.memory_allocated(device)
        torch.cuda.reset_peak_memory_stats(device)
        t0 = time.perf_counter()
        out = model(input_ids=prompt, attention_mask=torch.ones_like(prompt),
                    use_cache=True, return_dict=True, logits_to_keep=1)
        cache = out.past_key_values
        torch.cuda.synchronize(device)
        t1 = time.perf_counter()
        del out
        for token, mask in zip(token_tensors, masks):
            out = model(input_ids=token, attention_mask=mask, past_key_values=cache,
                        use_cache=True, return_dict=True, logits_to_keep=1)
            cache = out.past_key_values
            del out
        torch.cuda.synchronize(device)
        t2 = time.perf_counter()
        memory = cuda_memory_report(torch, device, start_alloc)
        info = inspect_cache(cache)
        del cache
        return {"prefill_ms": (t1-t0)*1000,
                "decode_ms_per_step": (t2-t1)*1000/(n-1),
                "total_ms": (t2-t0)*1000, "memory": memory, "cache": info}

    model.eval()
    with torch.inference_mode():
        for _ in range(2):
            one()
        runs = [one() for _ in range(s.benchmark_repeats)]
    return {"scope": "full model forward+dynamic KV; fixed tokens; no sampling/tokenization/load; not service TTFT",
            "prompt_tokens": prompt.shape[1], "predictions": n, "decode_calls": n-1,
            "trajectory_sha256": fingerprint(request["input_ids"] + [continuation[i % len(continuation)] for i in range(n-1)]),
            "warmup": 2, "repeats": len(runs), "runs": runs,
            "median_prefill_ms": statistics.median(r["prefill_ms"] for r in runs),
            "median_decode_ms_per_step": statistics.median(r["decode_ms_per_step"] for r in runs),
            "warning": "each mode runs in a fresh process; background GPU load/clock drift still matters; repeat and vary order before performance claims"}

## 9. 完整真实执行流程

每个模式启动一个独立进程，顺序执行：

```text
环境检查 → 固定tokenizer/输入 → 加载权重 → 固定前缀前向
→ 模块审计 → 生成路径预热 → 全部题目 → 可选固定工作量基准
→ 保存报告 → 进程退出
```

三种模型不会同时驻留。退出进程比在同一Notebook里反复 `del model` 更容易隔离持久引用与分配器状态。但它不能清除其他进程占用的显存。

加载耗时可能包含网络下载，只作为诊断，不作为三路计算速度排名。每题保存原始文本、输入输出ID、EOS/预算停止、显存和计时口径。

遇到OOM或运行异常会写manifest、保留已完成原始记录和固定分母，停止当前模式，父进程继续执行下一模式。不会回退到CPU或更小模型后仍使用原模式名字。

In [22]:
# %% 10. 每种模式独立进程加载；失败留下真实记录，不回退更小模型

def run_worker(mode: str, directory: str | Path, s: Settings) -> int:
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    items = select_items(make_suite(), s.split)
    records, condition = [], {"mode": mode, "quantization_plan": quantization_plan(mode)}
    manifest = {"format": LESSON_FORMAT, "kind": "real_model", "mode": mode,
                "status": "starting", "settings": asdict(s), "weights_loaded": False,
                "stage": "environment", "raw_records": 0}
    json_write(directory / "manifest.json", manifest)
    json_write(directory / "dataset.json", items)
    # worker目录是新建目录；不把旧raw响应混入新运行。
    (directory / "raw.jsonl").write_text("", encoding="utf-8")
    model = None
    try:
        validate_settings(s)
        torch, _, _, AutoModel, _, BnbConfig = require_runtime(mode, s.gpu, True)
        env = doctor(s.gpu)
        manifest["environment"] = env
        manifest["stage"] = "tokenizer"
        json_write(directory / "manifest.json", manifest)
        config, tokenizer, items, requests, gen_options, info = inspect_inputs(s)
        json_write(directory / "inspection.json", info)
        contract = fair_contract(s, info, env, items)
        condition.update(contract_sha256=fingerprint(contract), contract=contract)
        manifest["contract"] = contract
        quant = make_quant_config(mode, torch, BnbConfig)
        device = torch.device(f"cuda:{s.gpu}")
        torch.cuda.synchronize(device)
        gc.collect()
        torch.cuda.empty_cache()
        before = torch.cuda.memory_allocated(device)
        torch.cuda.reset_peak_memory_stats(device)
        manifest["stage"] = "model_loading"
        json_write(directory / "manifest.json", manifest)
        start = time.perf_counter()
        kwargs = {"revision": REVISION, "local_files_only": s.local_files_only,
                  "trust_remote_code": False, "use_safetensors": True,
                  "dtype": torch.float16, "device_map": {"": s.gpu},
                  "attn_implementation": "sdpa", "low_cpu_mem_usage": True}
        if quant is not None:
            kwargs["quantization_config"] = quant
        model = AutoModel.from_pretrained(MODEL_ID, **kwargs)
        # 不再调用model.half()/model.to()；量化模块已按device_map放置。
        model.eval()
        for p in model.parameters():
            p.requires_grad_(False)
        load_memory = cuda_memory_report(torch, device, before)
        manifest.update(weights_loaded=True, load_seconds_including_possible_download=time.perf_counter()-start)
        manifest["stage"] = "warmup_and_module_audit"
        json_write(directory / "manifest.json", manifest)
        # 一次固定前缀前向初始化延迟量化状态，也生成同前缀logits诊断文件。
        probe = fixed_prefix_probe(model, requests[0], s.gpu, directory / "probe_logits.pt")
        audit = audit_modules(model, mode, s.gpu)
        audit["load_memory"] = load_memory
        audit["fixed_prefix_probe"] = probe
        if quant is not None:
            audit["actual_quantization_config"] = quant.to_dict()
        json_write(directory / "model_audit.json", audit)
        # 生成路径独立预热一次，丢弃结果，不计入题集成绩。
        warm_options = {**gen_options, "max_new_tokens": 1}
        warm_s = Settings(**{**asdict(s), "max_new": 1})
        generate_request(model, tokenizer, requests[0], warm_options, warm_s)
        gc.collect()
        torch.cuda.empty_cache()
        manifest["post_warmup_allocated_bytes"] = torch.cuda.memory_allocated(device)
        manifest["stage"] = "evaluation"
        json_write(directory / "manifest.json", manifest)
        for request in requests:
            try:
                record = generate_request(model, tokenizer, request, gen_options, s)
            except Exception as error:
                record = {"id": request["id"], "status": "error",
                          "error": {"type": type(error).__name__, "message": str(error)}}
                records.append(record)
                with (directory / "raw.jsonl").open("a", encoding="utf-8") as f:
                    f.write(json.dumps(record, ensure_ascii=False, allow_nan=False) + "\n")
                # OOM/设备故障时不偷偷减少预算或重试挑最佳答案；剩余题保留为缺失。
                raise
            records.append(record)
            with (directory / "raw.jsonl").open("a", encoding="utf-8") as f:
                f.write(json.dumps(record, ensure_ascii=False, allow_nan=False) + "\n")
            print(f"{mode}: {len(records)}/{len(items)} {record['id']} {record['stop_reason']}", flush=True)
        if s.benchmark:
            manifest["stage"] = "benchmark"
            json_write(directory / "manifest.json", manifest)
            json_write(directory / "benchmark.json", benchmark_fixed_tokens(model, tokenizer, requests[0], s))
        report = build_report(items, records, kind="real_model", condition=condition,
                              scope="dev" if s.split=="dev" else "full_eval", complete=True)
        json_write(directory / "report.json", report)
        manifest.update(status="complete", stage="finished", raw_records=len(records),
                        audit_passed=True, benchmark_requested=s.benchmark)
        json_write(directory / "manifest.json", manifest)
        return 0
    except Exception as error:
        manifest.update(status="failed", raw_records=len(records),
                        error={"type": type(error).__name__, "message": str(error)},
                        traceback=traceback.format_exc())
        # 部分评分维持完整计划分母，但不放入'成功量化模型'排行榜。
        report = build_report(items, records, kind="real_model", condition=condition,
                              scope="dev" if s.split=="dev" else "full_eval", complete=False)
        json_write(directory / "report.json", report)
        json_write(directory / "manifest.json", manifest)
        print(f"{mode} failed at {manifest['stage']}: {error}", file=sys.stderr)
        return 2
    finally:
        if model is not None:
            del model
        gc.collect()

In [23]:
# %% 11. 配对比较：回归、固定前缀误差、显存与延迟分开呈现

def compare_artifacts(root: str | Path) -> dict:
    root = Path(root)
    modes, reports, manifests = {}, {}, {}
    for mode in MODES:
        d = root / mode
        if not (d / "manifest.json").exists():
            modes[mode] = {"status": "not_run"}
            continue
        manifest = json.loads((d / "manifest.json").read_text(encoding="utf-8"))
        manifests[mode] = manifest
        modes[mode] = {"status": manifest["status"], "error": manifest.get("error")}
        if manifest["status"] != "complete":
            continue
        report = json.loads((d / "report.json").read_text(encoding="utf-8"))
        verify_report(report)
        if not report["complete"] or report["kind"] != "real_model":
            raise ValueError("完成状态与报告类型不一致")
        if report["condition"].get("contract") != manifest["contract"]:
            raise ValueError("报告与manifest合同不一致")
        reports[mode] = report
        audit = json.loads((d / "model_audit.json").read_text(encoding="utf-8"))
        if not audit.get("passed"):
            raise ValueError("未通过模块审计")
        ok = [r for r in report["raw_records"] if r["status"] == "ok"]
        modes[mode].update(metrics=report["metrics"], module_counts=audit["counts"],
                           post_warmup_allocated_bytes=manifest.get("post_warmup_allocated_bytes"),
                           max_request_peak_bytes=max((r["memory"]["peak_allocated_bytes"] for r in ok), default=None),
                           caveat="task outputs may have different lengths; request time/peak not a fixed-work benchmark")
    comparisons = {}
    if "fp16" in reports:
        for mode in ("int8", "nf4"):
            if mode not in reports:
                continue
            require_matching_contract(manifests["fp16"]["contract"], manifests[mode]["contract"])
            comparison = paired_compare(reports["fp16"], reports[mode])
            # 同一题/同一前缀原始logits才能直接比较，不比较已经分叉的续写前缀。
            import torch
            a = torch.load(root / "fp16" / "probe_logits.pt", map_location="cpu", weights_only=True)
            b = torch.load(root / mode / "probe_logits.pt", map_location="cpu", weights_only=True)
            if a["input_ids"] != b["input_ids"] or a["request_id"] != b["request_id"]:
                raise ValueError("固定前缀诊断输入不一致")
            comparison["fixed_prefix_logits"] = logits_difference(a["logits"], b["logits"])
            left = root / "fp16" / "benchmark.json"
            right = root / mode / "benchmark.json"
            if left.exists() and right.exists():
                aa, bb = json.loads(left.read_text()), json.loads(right.read_text())
                if aa["trajectory_sha256"] != bb["trajectory_sha256"] or aa["predictions"] != bb["predictions"]:
                    raise ValueError("固定工作量基准不匹配")
                comparison["fixed_work_decode_ratio_fp16_over_variant"] = aa["median_decode_ms_per_step"] / bb["median_decode_ms_per_step"]
            comparisons[mode] = comparison
    result = {"format": LESSON_FORMAT, "modes": modes, "paired_vs_fp16": comparisons,
              "note": "failed/not_run are not model quality scores; same score can hide regressions; fixed 24-task classroom regression only"}
    json_write(root / "comparison.json", result)
    return result


def launch_experiment(s: Settings, mode="all", script_path: str | Path | None=None) -> Path:
    validate_settings(s)
    if mode not in (*MODES, "all"):
        raise ValueError("未知模式")
    path = Path(script_path) if script_path is not None else Path(__file__)
    if not path.is_file():
        raise ValueError("新进程入口脚本不存在；Notebook请使用内置导出单元")
    run_dir = unique_run_dir(s.output_dir, "real")
    json_write(run_dir / "settings.json", asdict(s))
    json_write(run_dir / "environment.json", doctor(s.gpu))
    sequence = MODES if mode == "all" else (mode,)
    exits = {}
    for item in sequence:
        directory = run_dir / item
        directory.mkdir()
        cmd = [sys.executable, str(path.resolve()), "--worker", item,
               "--settings-file", str((run_dir / "settings.json").resolve()),
               "--run-dir", str(directory.resolve())]
        print(f"运行独立进程：{item}；日志 → {directory / 'worker.log'}", flush=True)
        with (directory / "worker.log").open("w", encoding="utf-8") as log:
            done = subprocess.run(cmd, stdout=log, stderr=subprocess.STDOUT, check=False)
        exits[item] = done.returncode
        # 处理进程被系统终止、未能写入完整manifest的情况。
        manifest_path = directory / "manifest.json"
        current = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}
        if done.returncode != 0 and current.get("status") != "failed":
            current.update(format=LESSON_FORMAT, mode=item, status="failed",
                           error={"type": "WorkerProcessExit", "message": f"exit code {done.returncode}; see worker.log"})
            json_write(manifest_path, current)
    json_write(run_dir / "process_exit_codes.json", exits)
    result = compare_artifacts(run_dir)
    print(json.dumps({"directory": str(run_dir), "status": {k:v["status"] for k,v in result["modes"].items()}}, ensure_ascii=False, indent=2))
    return run_dir

### 9.1 默认关闭真实下载与生成

先设置 `RUN_REAL_INSPECTION=True` 只读分词器和配置。检查通过后再设 `RUN_REAL_EXPERIMENT=True`。

先用4道开发题，确认量化模块验收和格式，再固定配置运行24道评测题。生成记录包含明文，分享前检查私人信息。

In [24]:
RUN_REAL_INSPECTION = False
RUN_REAL_EXPERIMENT = False
REAL_MODE = "all"       # fp16 / int8 / nf4 / all
REAL_SPLIT = "dev"      # dev=4题；eval=24题
REAL_GPU = 0
RUN_FIXED_WORK_BENCHMARK = False
LOCAL_FILES_ONLY = False  # 首次缓存不完整时不能设True
REAL_RUN_DIRECTORY = None
REAL_SETTINGS = Settings(
    split=REAL_SPLIT, gpu=REAL_GPU, benchmark=RUN_FIXED_WORK_BENCHMARK,
    local_files_only=LOCAL_FILES_ONLY,
)

if RUN_REAL_INSPECTION:
    *_, inspected = inspect_inputs(REAL_SETTINGS)
    print({k: inspected[k] for k in ("model", "revision", "input_ids_sha256", "weights_loaded")})
else:
    print("真实分词器检查未开启：没有下载，没有权重加载。")

真实分词器检查未开启：没有下载，没有权重加载。


### 9.2 Notebook自带独立进程入口

下一单元折叠保存了与配套 `.py` **逐字节一致**的压缩源码。仅当开启真实实验时，才将它导出到 `lesson_31_workspace`，再由同一Python解释器启动。

这让Notebook不依赖你另外下载 `.py` 文件；它不是从网络执行未知脚本。导出时会校验内置SHA-256，已有同名但内容不同的文件会被拒绝覆盖。

In [25]:
import base64
import gzip

_WORKER_B64 = (
    '...'
)
_WORKER_SHA256 = 'e259b50e360ed56e52cdec93dc5aeba6766247ed8cbdfde74bc571fce6f30510'

def export_worker(path):
    raw = gzip.decompress(base64.b64decode(_WORKER_B64))
    if hashlib.sha256(raw).hexdigest() != _WORKER_SHA256:
        raise ValueError("内置worker校验失败")
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and path.read_bytes() != raw:
        raise ValueError("已有不同源码，不覆盖；请换一个workspace路径")
    path.write_bytes(raw)
    return path

if RUN_REAL_EXPERIMENT:
    worker = export_worker(Path("lesson_31_workspace") / "lesson_31_real_quantized_loading_and_regression.py")
    REAL_RUN_DIRECTORY = launch_experiment(REAL_SETTINGS, REAL_MODE, script_path=worker)
    print("实际运行目录：", REAL_RUN_DIRECTORY)
else:
    print("真实模型实验未开启：没有导出worker，没有CUDA/模型运行。")

真实模型实验未开启：没有导出worker，没有CUDA/模型运行。


In [26]:
if REAL_RUN_DIRECTORY is not None:
    comparison = compare_artifacts(REAL_RUN_DIRECTORY)
    print(json.dumps(comparison, ensure_ascii=False, indent=2))
else:
    print("没有真实运行目录；不展示模型质量、速度或显存数字。")

没有真实运行目录；不展示模型质量、速度或显存数字。


## 10. 命令行入口与输出文件

```bash
# 只检查环境，不联网
python lesson_31_real_quantized_loading_and_regression.py --doctor

# 标准库/PyTorch离线检查
python lesson_31_real_quantized_loading_and_regression.py --checks-only

# 只下载/检查配套tokenizer，不加载权重
python lesson_31_real_quantized_loading_and_regression.py --inspect --split dev

# 三路开发题；限定单张CUDA，不自动回退CPU
python lesson_31_real_quantized_loading_and_regression.py --run --mode all --split dev

# 设置确定后跑24题
python lesson_31_real_quantized_loading_and_regression.py --run --mode all --split eval

# 单独加载NF4；不自动制造与FP16的对照结果
python lesson_31_real_quantized_loading_and_regression.py --run --mode nf4 --split dev

# 正式三路并加固定工作量基准
python lesson_31_real_quantized_loading_and_regression.py --run --mode all --split eval --benchmark
```

完整缓存已有后加 `--local-files-only`。其他卡用 `--gpu 1` 等。所有依赖版本会记录，不更改模型来源。

```text
lesson_31_outputs/real_时间戳/
    settings.json
    environment.json
    process_exit_codes.json
    comparison.json
    fp16/、int8/、nf4/
        worker.log
        manifest.json
        dataset.json
        inspection.json
        model_audit.json
        probe_logits.pt
        raw.jsonl
        report.json
        benchmark.json   # 仅明确开启且成功完成时
```

每次创建新目录，不覆盖之前实验。加载失败时只会有失败前实际生成的文件。
`probe_logits.pt` 是本次固定提示的数值诊断，不是模型权重文件；只读取你信任的本课产物。

重新比较一个实际运行目录，无需再次调用GPU：

```bash
python lesson_31_real_quantized_loading_and_regression.py --compare lesson_31_outputs/real_实际时间戳
```

不要把 `eval` 题不断用于调参后，仍称它是独立盲测。本课是小规模课堂回归题集，不是排行榜。

## 11. 独立练习A：组装NF4配置

实现 `nf4_config_ex(torch_module, factory)`，返回factory创建的配置：NF4、双重量化、FP16计算、uint8存储、禁止CPU卸载、跳过lm_head。

练习不需要真实bitsandbytes。检查器用捕获关键字参数的factory验证配置；真实执行时对应 `BitsAndBytesConfig`。

不能把NF4改成均匀INT4，也不能用整网 `.half()` 来完成此题。

In [27]:
def nf4_config_ex(torch_module, factory):
    raise NotImplementedError("请补全NF4配置")


def check_exercise_a(fn):
    from types import SimpleNamespace
    fake_torch = SimpleNamespace(float16=object(), uint8=object())
    got = fn(fake_torch, lambda **kw: kw)
    expected = make_quant_config("nf4", fake_torch, lambda **kw: kw)
    assert got == expected, (got, expected)
    assert got["bnb_4bit_compute_dtype"] is fake_torch.float16
    print("练习A通过（配置接口检查，不是CUDA内核检查）。")

RUN_EXERCISE_A = False
if RUN_EXERCISE_A:
    check_exercise_a(nf4_config_ex)

## 12. 独立练习B：显存口径

实现 `summarize_memory_ex(start, live, peak, reserved)`，返回与 `memory_numbers` 相同的字段；输入均为非负整数字节数，不接受bool，拒绝不可能的峰值关系。

重点是 `peak_increment_bytes = peak - start`，不是 `reserved - live`。不存在CUDA数据时调用侧应使用 `None` 或明确跳过，不把缺测填成0。

In [28]:
def summarize_memory_ex(start, live, peak, reserved):
    raise NotImplementedError("请补全显存摘要")


def check_exercise_b(fn):
    for values in ((100,110,180,200), (0,0,0,0), (200,180,250,300)):
        assert fn(*values) == memory_numbers(*values)
    for values in ((100,150,120,200), (-1,0,0,0), (True,1,1,1), (0,4,4,2)):
        try:
            fn(*values)
        except ValueError:
            pass
        else:
            raise AssertionError(f"应当拒绝：{values}")
    print("练习B通过。")

RUN_EXERCISE_B = False
if RUN_EXERCISE_B:
    check_exercise_b(summarize_memory_ex)

## 13. 独立练习C：按ID定位回归

实现 `paired_success_ex(items, before_raw, after_raw)`，先使用严格评分器，再按ID找出修好和退步的题，返回字段与 `paired_success_answer` 一致。

响应顺序可以变化，不能直接按列表位置配对。错误和缺失题保留在计划分母，重复ID必须报错。

In [29]:
def paired_success_ex(items, before_raw, after_raw):
    raise NotImplementedError("请补全配对回归分析")


def check_exercise_c(fn):
    fixture = regression_fixture()
    items = fixture["before"]["items"]
    a, b = fixture["before"]["raw_records"], fixture["after"]["raw_records"]
    expected = paired_success_answer(items, a, b)
    assert fn(items, a, list(reversed(b))) == expected
    missing = b[:-1]
    assert fn(items, a, missing) == paired_success_answer(items,a,missing)
    try:
        fn(items, a, b+[b[0]])
    except ValueError:
        pass
    else:
        raise AssertionError("重复响应不应被接受")
    print("练习C通过。")

RUN_EXERCISE_C = False
if RUN_EXERCISE_C:
    check_exercise_c(paired_success_ex)

---
## 14. 参考答案：先完成练习，再展开

参考函数使用新名字，不覆盖你的实现。检查只证明这些配置/规则用例通过，不代替真实GPU验收。

In [30]:
def nf4_config_reference(torch_module, factory):
    return factory(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch_module.float16,
        bnb_4bit_quant_storage=torch_module.uint8,
        llm_int8_skip_modules=["lm_head"],
        llm_int8_enable_fp32_cpu_offload=False,
    )


def summarize_memory_reference(start, live, peak, reserved):
    if any(type(x) is not int or x < 0 for x in (start, live, peak, reserved)):
        raise ValueError("必须是非负整数字节数")
    if peak < max(start, live) or reserved < live:
        raise ValueError("显存关系不可能")
    return {
        "start_allocated_bytes": start,
        "end_allocated_bytes": live,
        "peak_allocated_bytes": peak,
        "peak_increment_bytes": peak-start,
        "end_reserved_bytes": reserved,
        "scope": "PyTorch CUDA allocator; not full process / nvidia-smi",
    }


def paired_success_reference(items, before_raw, after_raw):
    before = score_records(items, before_raw)
    after = score_records(items, after_raw)
    a = {r["id"]:r["success"] for r in before["graded"]}
    b = {r["id"]:r["success"] for r in after["graded"]}
    return {
        "n":len(items),
        "before":sum(a.values()),
        "after":sum(b.values()),
        "fixed_ids":[key for key in a if not a[key] and b[key]],
        "regressed_ids":[key for key in a if a[key] and not b[key]],
    }

check_exercise_a(nf4_config_reference)
check_exercise_b(summarize_memory_reference)
check_exercise_c(paired_success_reference)

练习A通过（配置接口检查，不是CUDA内核检查）。
练习B通过。
练习C通过。


## 15. 自动检查与本课验收

先运行本课离线检查。它们覆盖：题集指纹、配置、量化清单、设备回退、EOS、输出切片、失败分母、配对规则、显存口径、固定前缀误差和缓存接口。

**真实验收还需在你的电脑上完成：** 模块审计通过；同源FP16/INT8/NF4跑完开发题；固定设置后完成24题；能解释回归题和三种显存口径。若依赖或显卡执行失败，保留失败阶段，不捏造模型成绩。

In [31]:
# %% 13. 离线固定检查：均为协议/接口/数学检查，不是bnb CUDA内核测试

def run_checks(verbose: bool = True) -> dict:
    passed, skipped = [], []

    def check(name, predicate):
        result = predicate()
        if not result:
            raise AssertionError(name)
        passed.append(name)

    def rejects(fn, error_type=ValueError):
        try:
            fn()
        except error_type:
            return True
        return False

    def changed(settings=None, **kwargs):
        return Settings(**{**asdict(settings or Settings()), **kwargs})

    suite = make_suite()
    dev, items = select_items(suite, "dev"), select_items(suite, "eval")
    check("第29课完整题集指纹不变", lambda: fingerprint(suite) == EXPECTED_SUITE_SHA256)
    check("第29课评测协议指纹不变", lambda: protocol_for(items)["protocol_sha256"] == EXPECTED_EVAL_PROTOCOL_SHA256)
    check("4道开发题", lambda: len(dev) == 4)
    check("24道评测题", lambda: len(items) == 24)
    check("四类各6道", lambda: set(Counter(x["task"] for x in items).values()) == {6})
    check("请求不含答案", lambda: all(set(public_request(x)) == {"id", "messages"} for x in suite))
    changed_gold = copy.deepcopy(items[0]); changed_gold["expected"] += 1
    check("修改答案不改变请求", lambda: public_request(changed_gold) == public_request(items[0]))
    check("修改答案改变协议", lambda: protocol_for([changed_gold]) != protocol_for([items[0]]))
    check("合法默认配置", lambda: validate_settings(Settings()) is None)
    for key, value in [("max_new", 0), ("max_input", True), ("gpu", -1), ("seed", -1),
                       ("benchmark_steps", 1), ("benchmark_repeats", 0), ("split", "test")]:
        check(f"拒绝非法设置{key}", lambda k=key,v=value: rejects(lambda: validate_settings(changed(**{k:v}))))
    check("拒绝总预算超限", lambda: rejects(lambda: validate_settings(changed(context_limit=64))))
    check("FP16不启用量化配置", lambda: quantization_plan("fp16") is None)
    check("INT8加载开关", lambda: quantization_plan("int8")["load_in_8bit"] is True)
    check("INT8关闭CPU卸载", lambda: quantization_plan("int8")["llm_int8_enable_fp32_cpu_offload"] is False)
    check("INT8阈值显式固定", lambda: quantization_plan("int8")["llm_int8_threshold"] == 6.0)
    check("NF4不是均匀INT4", lambda: quantization_plan("nf4")["bnb_4bit_quant_type"] == "nf4")
    check("NF4计算使用FP16", lambda: quantization_plan("nf4")["bnb_4bit_compute_dtype"] == "float16")
    check("NF4双重量化启用", lambda: quantization_plan("nf4")["bnb_4bit_use_double_quant"] is True)
    check("NF4用uint8存储打包码", lambda: quantization_plan("nf4")["bnb_4bit_quant_storage"] == "uint8")
    check("输出头显式不转换", lambda: all(quantization_plan(m)["llm_int8_skip_modules"] == ["lm_head"] for m in ("int8", "nf4")))
    check("未知量化模式报错", lambda: rejects(lambda: quantization_plan("int4")))
    altered = quantization_plan("nf4"); altered["llm_int8_skip_modules"].append("evil")
    check("配置返回独立副本", lambda: quantization_plan("nf4")["llm_int8_skip_modules"] == ["lm_head"])
    cfg = generation_options(Settings(), [9, 10, 9], 0)
    check("所有EOS保留并去重", lambda: cfg["eos_token_id"] == [9,10])
    check("固定贪心不继承采样", lambda: cfg["do_sample"] is False and "temperature" not in cfg)
    check("明确动态未量化缓存", lambda: cfg["cache_implementation"] == "dynamic")
    check("拒绝空EOS", lambda: rejects(lambda: generation_options(Settings(), [], 0)))
    check("拒绝bool特殊ID", lambda: rejects(lambda: generation_options(Settings(), [True], 0)))
    check("正确裁掉提示", lambda: split_generated_ids([3,4,8,9], [3,4], [9], 4)["visible_ids"] == [8])
    check("第二个EOS正常停止", lambda: split_generated_ids([3,8,10], [3], [9,10], 4)["stop_reason"] == "eos")
    check("预算结束区别于EOS", lambda: split_generated_ids([3,8,7], [3], [9], 2)["stop_reason"] == "length")
    check("拒绝前缀不一致", lambda: rejects(lambda: split_generated_ids([4,8], [3], [9], 2)))
    check("拒绝超过预算", lambda: rejects(lambda: split_generated_ids([3,8,7], [3], [9], 1)))
    check("负号不可丢失", lambda: parse_answer("arithmetic", "-4") == (-4, None))
    check("不从解释找答案", lambda: parse_answer("arithmetic", "答案是45")[1] is not None)
    check("不接受重复JSON键", lambda: rejects(lambda: strict_json_loads('{"x":1,"x":2}')))
    check("JSON bool不是整数", lambda: parse_answer("extraction", '{"item":"红茶","quantity":true}')[1] is not None)
    check("NaN不是合法JSON答案", lambda: rejects(lambda: strict_json_loads('{"x":NaN}')))
    gold = reference_success_records(items)
    check("完美手工夹具24题", lambda: score_records(items,gold)["metrics"]["success"] == 24)
    check("缺失响应分母不缩小", lambda: score_records(items,gold[:-1])["metrics"]["n"] == 24)
    bad = copy.deepcopy(gold); bad[0] = {"id":items[0]["id"],"status":"error","error":{"type":"fixture"}}
    check("运行错误计入原分母", lambda: score_records(items,bad)["metrics"]["success"] == 23)
    length = copy.deepcopy(gold); length[0]["stop_reason"] = "length"
    check("截断正确答案与严格成功区分", lambda: score_records(items,length)["metrics"]["answer_correct"] == 24 and score_records(items,length)["metrics"]["success"] == 23)
    check("拒绝重复题目响应", lambda: rejects(lambda: score_records(items,gold+[gold[0]])))
    f = regression_fixture()
    check("同分数仍有回归", lambda: f["paired"]["before_success"] == f["paired"]["after_success"] == 23)
    check("定位修好题", lambda: f["paired"]["fixed_ids"] == [items[0]["id"]])
    check("定位退步题", lambda: f["paired"]["regressed_ids"] == [items[1]["id"]])
    incomplete = copy.deepcopy(f["after"]); incomplete["complete"] = False
    check("拒绝把未完成运行排名", lambda: rejects(lambda: paired_compare(f["before"],incomplete)))
    forged = copy.deepcopy(f["after"]); forged["metrics"]["success"] = 24
    check("拒绝篡改汇总成绩", lambda: rejects(lambda: verify_report(forged)))
    check("对照合同相同可比较", lambda: require_matching_contract({"x":1},{"x":1}) is None)
    for field in ("revision", "input_ids_sha256", "generation_options", "gpu", "packages"):
        check("检测条件变化："+field, lambda f=field: rejects(lambda: require_matching_contract({f:1},{f:2})))
    check("显存峰值差正确", lambda: memory_numbers(100,110,180,200)["peak_increment_bytes"] == 80)
    check("显存reserved不是allocated", lambda: memory_numbers(100,110,180,200)["end_reserved_bytes"] == 200)
    check("拒绝错误显存顺序", lambda: rejects(lambda: memory_numbers(100,150,120,200)))
    check("拒绝负显存值", lambda: rejects(lambda: memory_numbers(-1,0,0,0)))
    check("两层有14个投影", lambda: len(required_projections(2)) == 14)
    check("所有参数指定GPU", lambda: assert_single_gpu_devices({"w":"cuda:0"},0) is None)
    for wrong in ("cpu", "disk", "meta", "cuda:1"):
        check("拒绝设备回退"+wrong, lambda d=wrong: rejects(lambda: assert_single_gpu_devices({"w":d},0)))
    rows = [{"name":n,"kind":"Linear4bit","weight_dtype":"torch.uint8","has_quant_state":True,
             "quant_type":"nf4","compute_dtype":"torch.float16","nested":True}
            for n in required_projections(1)]
    check("合法NF4清单通过", lambda: validate_projection_inventory(rows,"nf4",1) is None)
    for key, value in [("kind","Linear"),("weight_dtype","torch.float16"),("has_quant_state",False),
                       ("quant_type","fp4"),("nested",False),("compute_dtype","torch.float32")]:
        mutated = copy.deepcopy(rows); mutated[0][key] = value
        check("捕获量化伪成功："+key, lambda r=mutated: rejects(lambda: validate_projection_inventory(r,"nf4",1)))
    check("漏转换层不通过", lambda: rejects(lambda: validate_projection_inventory(rows[:-1],"nf4",1)))
    check("参数量不是除四即全显存", lambda: ideal_storage_demo()["four_bit_codes_plus_fp16_rest_bytes"] > ideal_storage_demo()["fp16_weight_bytes"] / 4)
    # 以下用真实PyTorch张量检查数学/缓存接口；不加载Transformers或bnb权重。
    torch_checks_start = len(passed)
    try:
        import torch
        from types import SimpleNamespace
        factory = lambda **kwargs: kwargs
        b = make_quant_config("nf4", torch, factory)
        check("配置工厂得到真实torch.dtype", lambda: b["bnb_4bit_compute_dtype"] is torch.float16 and b["bnb_4bit_quant_storage"] is torch.uint8)
        check("FP16不构造BitsAndBytesConfig", lambda: make_quant_config("fp16", torch, lambda **k: (_ for _ in ()).throw(AssertionError())) is None)
        check("旧式KV实际载荷", lambda: inspect_cache(((torch.zeros(1,2,3,4),torch.zeros(1,2,3,4)),))["payload_bytes"] == 192)
        cache = SimpleNamespace(layers=[SimpleNamespace(keys=torch.zeros(1,2,3,4,dtype=torch.float16),
                                                         values=torch.zeros(1,2,3,4,dtype=torch.float16))])
        check("动态KV载荷和FP16类型", lambda: inspect_cache(cache)["payload_bytes"] == 96 and inspect_cache(cache)["dtypes"] == ["torch.float16"])
        check("缺失缓存不伪装零字节", lambda: inspect_cache(None)["payload_bytes"] is None)
        check("未知缓存接口明确标注", lambda: inspect_cache(object())["available"] is False)
        x = torch.tensor([0.501,0.499])
        check("相同logits误差为零", lambda: logits_difference(x,x)["max_abs"] == 0)
        check("微小误差可反转top1", lambda: not logits_difference(x,x.flip(0))["top1_same"])
        check("正向KL非负", lambda: logits_difference(x,x.flip(0))["kl_fp16_to_variant"] >= 0)
        check("拒绝不同形状logits", lambda: rejects(lambda: logits_difference(x,torch.zeros(3))))
        check("拒绝非有限logits", lambda: rejects(lambda: logits_difference(x,torch.tensor([float('inf'),0.]))))
        # 仅测试对照目录分类，不制造真实模型成功的假报告。
        with tempfile.TemporaryDirectory() as td:
            d=Path(td); (d/"nf4").mkdir()
            json_write(d/"nf4"/"manifest.json", {"status":"failed","error":{"type":"fixture"}})
            comp=compare_artifacts(d)
            check("失败模式没有质量分数", lambda: comp["modes"]["nf4"]["status"] == "failed" and "metrics" not in comp["modes"]["nf4"])
            check("缺少FP16不制造配对结果", lambda: comp["paired_vs_fp16"] == {})
    except ImportError:
        skipped.append("PyTorch张量/配置接口检查：未安装torch")
    torch_count = len(passed) - torch_checks_start
    result = {"passed":len(passed),"torch_interface_checks":torch_count,
              "names":passed,"skipped":skipped,"real_bnb_cuda_or_qwen_tested":False}
    if verbose:
        print(f"离线固定检查通过：{len(passed)} 项，其中 PyTorch数学/接口检查 {torch_count} 项。")
        print("这不是实际 bitsandbytes CUDA 或 Qwen 权重测试。")
    return result

In [32]:
CHECK_RESULTS = run_checks()
print("已通过：", CHECK_RESULTS["passed"])
print("其中PyTorch数学/接口：", CHECK_RESULTS["torch_interface_checks"])
print("跳过：", CHECK_RESULTS["skipped"])
print("真实bnb CUDA/Qwen已测试：", CHECK_RESULTS["real_bnb_cuda_or_qwen_tested"])

离线固定检查通过：91 项，其中 PyTorch数学/接口检查 13 项。
这不是实际 bitsandbytes CUDA 或 Qwen 权重测试。
已通过： 91
其中PyTorch数学/接口： 13
跳过： []
真实bnb CUDA/Qwen已测试： False


## 16. 常见故障怎么定位

| 现象 | 先查看什么 | 不要怎样处理 |
|---|---|---|
| 找不到CUDA | `--doctor` 中PyTorch构建、cuda_available、gpu | 不把CPU结果贴成3060结果 |
| bnb导入/内核失败 | worker.log、驱动/torch/bnb版本、官方安装表 | 不自动换另一套后端后仍叫同一实验 |
| OOM | 是否有旧内核驻留、请求长度、load vs generate失败阶段 | 不仅给失败分支缩短预算继续比较 |
| NF4没有Linear4bit | model_audit/量化配置是否真正传入 | 不只看 load_in_4bit 这个布尔标志 |
| 总分相同 | fixed_ids、regressed_ids和原始文本 | 不宣称每题行为都没变 |
| 四位速度反而慢 | 是否同工作量、预热、同步、输出长度、GPU负载 | 不把位宽当成速度保证 |

这节课不保存新的量化模型Hub仓库，不做训练，也不执行生成代码。成功跑通后冻结环境版本和报告，作为后续LoRA/QLoRA对照的基础。

**下一课：第32课——LoRA的低秩更新、冻结基座与从零实现。** 从可解释的小矩阵实验开始，理解“只训练少量参数”在数学和工程上意味着什么。

## 资料与实现依据

本课采用固定4.57系列接口，文档的最新版本可能不同；以报告记录的实际版本为准。

1. Hugging Face Transformers 4.57.3：[bitsandbytes集成与参数](https://huggingface.co/docs/transformers/v4.57.3/en/quantization/bitsandbytes)。
2. 原论文：[LLM.int8()](https://arxiv.org/abs/2208.07339)。
3. 原论文：[QLoRA：NF4、双重量化与低秩适配](https://arxiv.org/abs/2305.14314)。
4. bitsandbytes：[Linear4bit接口](https://huggingface.co/docs/bitsandbytes/reference/nn/linear4bit)。
5. PyTorch：[CUDA计时与内存管理](https://docs.pytorch.org/docs/stable/notes/cuda.html)。
6. Qwen：[本课固定commit的模型卡](https://huggingface.co/Qwen/Qwen3-0.6B/blob/c1899de289a04d12100db370d81485cdf75e47ca/README.md)。
7. bitsandbytes：[安装指南与硬件要求](https://huggingface.co/docs/bitsandbytes/installation)。
8. Transformers：[GenerationConfig与缓存设置](https://huggingface.co/docs/transformers/v4.57.3/en/main_classes/text_generation)。

已保存的Notebook输出只来自实际离线执行。教学夹具、公式估算、真实模型测量在字段与文字中分别标记。